In [ ]:
import itertools
import pandas as pd
import gc

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List

from common import *

In [ ]:
main_dir = "/kaggle/input/home-credit-credit-risk-model-stability/parquet_files/"
main_dir = "data/home-credit-credit-risk-model-stability/parquet_files/"
main_dir_train = f"{main_dir}train/"
main_dir_test= f"{main_dir}test/"

Base

Base - Train

In [ ]:
train_base = pd.read_parquet(f"{main_dir_train}train_base.parquet")

# Preprocess
convert_date_columns(train_base, ["date_decision"])

# train_base = train_base.drop(columns=["MONTH"])

Base - Test

In [ ]:
test_base = pd.read_parquet(f"{main_dir_test}test_base.parquet")

# Preprocess
convert_date_columns(test_base, ["date_decision"])

# test_base = test_base.drop(columns=["MONTH"])

Credit Bureau

In [ ]:
credit_bureau_names = ["train_credit_bureau_a_1_0", "train_credit_bureau_a_1_1",
                       "train_credit_bureau_a_1_2", "train_credit_bureau_a_1_3"]

train_credit_bureau_a_1 = []
categorical_columns = []
date_columns = ['dateofcredend_289D', 'dateofcredend_353D', 'dateofcredstart_181D',
                'dateofcredstart_739D', 'dateofrealrepmt_138D', 'lastupdate_1112D',
                'lastupdate_388D', 'refreshdate_3813885D', 'numberofoverdueinstlmaxdat_148D',
                'numberofoverdueinstlmaxdat_641D']
deleted_categorical = []


for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_train}{credit_bureau}.parquet")

    # Preprocess
    convert_date_columns(data, date_columns)

    # Categorical dataset
    many_categories = select_categorical_with_many_categories(data, 20)
    deleted_categorical = list(set(deleted_categorical).union(set(many_categories)))
    data = data.drop(columns=deleted_categorical)
    categorical_columns_df = get_columns_by_datatype(data, "categorical").tolist()

    # data = data.replace({"a55475b1":pd.NA})

    # do something with those ones which have many categories
    if categorical_columns != categorical_columns_df:
        categorical_columns = categorical_columns_df
    data = categorical_to_dummies(data, dtype=16)


    # Date
    # data_date = get_columns_by_datatype(data, "date")
    

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                            exclude_columns=["num_group2"])
    data_numerical["entries_credit_bureau_1"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min", "mean"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_1_max":"entries_credit_bureau_1"}).drop(columns=["entries_credit_bureau_1_min", "entries_credit_bureau_1_mean"])
    
    train_credit_bureau_a_1.append(data_numerical)
    # train_credit_bureau_a_1.append(data_numerical.iloc[[0], :])

    del data
    gc.collect()
    del data_numerical
    gc.collect()

train_credit_bureau_a_1 = pd.concat(train_credit_bureau_a_1)
train_credit_bureau_a_1 = train_credit_bureau_a_1.drop(columns=train_credit_bureau_a_1.columns.intersection(deleted_categorical))
train_credit_bureau_a_1 = train_credit_bureau_a_1.drop(columns=select_categorical_with_many_categories(train_credit_bureau_a_1, original_from_dummies=categorical_columns))
train_credit_bureau_a_1 = reduce_column_size(train_credit_bureau_a_1)
train_credit_bureau_a_1 = delete_constant_columns(train_credit_bureau_a_1)
train_credit_bureau_a_1 = delete_null_columns(train_credit_bureau_a_1, 0.4)

columns_credit_bureau_a_1 = train_credit_bureau_a_1.columns

In [ ]:
credit_bureau_names = ["test_credit_bureau_a_1_0", "test_credit_bureau_a_1_1",
                       "test_credit_bureau_a_1_2", "test_credit_bureau_a_1_3",
                       "test_credit_bureau_a_1_4"]

test_credit_bureau_a_1, credit_bureau_a_2_c = [], []
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_test}{credit_bureau}.parquet")

    # Preprocess
    convert_date_columns(data, date_columns)

    data = data.drop(columns=deleted_categorical)
    data = categorical_to_dummies(data, dtype=16)


    # Date
    # data_date = get_dataset_by_datatype(data, "date", extra_columns=["case_id"])

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                            exclude_columns=["num_group2"])
    data_numerical["entries_credit_bureau_1"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min", "mean"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_1_max":"entries_credit_bureau_1"}).drop(columns=["entries_credit_bureau_1_min", "entries_credit_bureau_1_mean"])

    
    test_credit_bureau_a_1.append(data_numerical)
    del data_numerical
    gc.collect()

test_credit_bureau_a_1 = pd.concat(test_credit_bureau_a_1)
test_credit_bureau_a_1 = test_credit_bureau_a_1[test_credit_bureau_a_1.columns.intersection(columns_credit_bureau_a_1)]
test_credit_bureau_a_1 = test_credit_bureau_a_1.reindex(columns=columns_credit_bureau_a_1)
test_credit_bureau_a_1 = test_credit_bureau_a_1[columns_credit_bureau_a_1]

Credit Bureau - 

In [ ]:
credit_bureau_names = ["train_credit_bureau_a_2_0", "train_credit_bureau_a_2_1",
                       "train_credit_bureau_a_2_2", "train_credit_bureau_a_2_3",
                       "train_credit_bureau_a_2_4", "train_credit_bureau_a_2_5",
                       "train_credit_bureau_a_2_6", "train_credit_bureau_a_2_7",
                       "train_credit_bureau_a_2_8", "train_credit_bureau_a_2_9",
                       "train_credit_bureau_a_2_10"]

categorical_columns = []
date_columns = ['dateofcredend_289D', 'dateofcredend_353D', 'dateofcredstart_181D',
                'dateofcredstart_739D', 'dateofrealrepmt_138D', 'lastupdate_1112D',
                'lastupdate_388D', 'refreshdate_3813885D', 'numberofoverdueinstlmaxdat_148D',
                'numberofoverdueinstlmaxdat_641D']
deleted_categorical = []

train_credit_bureau_a_2 = []
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_train}{credit_bureau}.parquet")

    # Preprocess
    convert_date_columns(data, [x for x in data.columns if "date" in x])

    # Categorical dataset
    many_categories = select_categorical_with_many_categories(data, 20)
    deleted_categorical = list(set(deleted_categorical).union(set(many_categories)))
    data = data.drop(columns=deleted_categorical)
    categorical_columns_df = get_columns_by_datatype(data, "categorical").tolist()

    # data = data.replace({"a55475b1":pd.NA})

    # do something with those ones which have many categories
    if categorical_columns != categorical_columns_df:
        categorical_columns = categorical_columns_df
    data = categorical_to_dummies(data, dtype=16)

    # Date
    # data_date = get_dataset_by_datatype(data, "date", extra_columns=["case_id"])

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                             exclude_columns=["num_group2"])
    del data
    gc.collect()
    data_numerical["entries_credit_bureau"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_max":"entries_credit_bureau"}).drop(columns=["entries_credit_bureau_min"])
    data_numerical = data_numerical.reset_index()


    
    train_credit_bureau_a_2.append(data_numerical)
    del data_numerical
    gc.collect()

train_credit_bureau_a_2 = pd.concat(train_credit_bureau_a_2)
train_credit_bureau_a_2 = train_credit_bureau_a_2.drop(columns=train_credit_bureau_a_2.columns.intersection(deleted_categorical))
train_credit_bureau_a_2 = train_credit_bureau_a_2.drop(columns=select_categorical_with_many_categories(train_credit_bureau_a_2, original_from_dummies=categorical_columns))
train_credit_bureau_a_2 = delete_constant_columns(train_credit_bureau_a_2)
train_credit_bureau_a_2 = delete_null_columns(train_credit_bureau_a_2, 0.1)

columns_credit_bureau_a_2 = train_credit_bureau_a_2.columns

In [ ]:
credit_bureau_names = ["test_credit_bureau_a_2_0", "test_credit_bureau_a_2_1",
                       "test_credit_bureau_a_2_2", "test_credit_bureau_a_2_3",
                       "test_credit_bureau_a_2_4", "test_credit_bureau_a_2_5",
                       "test_credit_bureau_a_2_6", "test_credit_bureau_a_2_7",
                       "test_credit_bureau_a_2_8", "test_credit_bureau_a_2_9",
                       "test_credit_bureau_a_2_10", "test_credit_bureau_a_2_11"]

test_credit_bureau_a_2, credit_bureau_a_2_c = [], []
for credit_bureau in credit_bureau_names:
    data = pd.read_parquet(f"{main_dir_test}{credit_bureau}.parquet")

    # Preprocess
    convert_date_columns(data, [x for x in data.columns if "date" in x])

    data = data.drop(columns=deleted_categorical)
    data = categorical_to_dummies(data, dtype=16)

    # Date
    # data_date = get_dataset_by_datatype(data, "date", extra_columns=["case_id"])

    # Numerical dataset
    data_numerical = get_dataset_by_datatype(data, "numerical", extra_columns=["case_id"],
                                             exclude_columns=["num_group2"])
    del data
    gc.collect()
    data_numerical["entries_credit_bureau"] = data_numerical.groupby(["case_id"])["num_group1"].transform("count")
    data_numerical = data_numerical.drop(columns=["num_group1"])
    data_numerical = data_numerical.groupby(["case_id"]).agg(["max", "min"])
    data_numerical.columns = ["_".join(x) for x in data_numerical.columns]
    data_numerical = data_numerical.rename(columns={"entries_credit_bureau_max":"entries_credit_bureau"}).drop(columns=["entries_credit_bureau_min"])
    data_numerical = data_numerical.reset_index()


    
    test_credit_bureau_a_2.append(data_numerical)
    del data_numerical
    gc.collect()

test_credit_bureau_a_2 = pd.concat(test_credit_bureau_a_2)
test_credit_bureau_a_2 = test_credit_bureau_a_2[test_credit_bureau_a_2.columns.intersection(columns_credit_bureau_a_2)]
test_credit_bureau_a_2 = test_credit_bureau_a_2.reindex(columns=columns_credit_bureau_a_2)
test_credit_bureau_a_2 = test_credit_bureau_a_2[columns_credit_bureau_a_2]

Person

In [ ]:
train_person_1 = read_sets_of_dataframes(main_dir_train, ["train_person_1"])[0]
test_person_1 = read_sets_of_dataframes(main_dir_test, ["test_person_1"])[0]

Person - Training data

In [ ]:
# Only choosing the info related to the person
train_person_1 = train_person_1[train_person_1["num_group1"]==0]

# Preprocess
train_person_1 = reduce_column_size(train_person_1)
convert_date_columns(train_person_1, ["birth_259D", "birthdate_87D"])
# train_person_1 = train_person_1.replace({"a55475b1": pd.NA})

# Feature selection
train_person_1 = delete_null_columns(train_person_1, 0.1, 
                                     cols_to_exclude = ["case_id", "education_927M"])
train_person_1["incometype_1044T"] = train_person_1["incometype_1044T"].map(lambda x: "HANDICAPPED"
                                                                            if "HANDIC" in x else x)

train_person_1 = train_person_1.merge(train_base[["case_id", "target"]], on=["case_id"], how="left")
train_person_1 = delete_constant_columns(train_person_1)
train_person_1 = select_by_chi2(train_person_1)
train_person_1 = train_person_1.drop(columns=["target"])

person_columns = train_person_1.columns

# Feature engineering
train_person_1 = transform_location_info(train_person_1) ################ check to transform all (?)

# Output
train_person_1 = order_columns_alphabetically(train_person_1)
train_person_1.head()

Person - Test data

In [ ]:
# Only choosing the info related to the person
test_person_1 = test_person_1[test_person_1["num_group1"]==0]

# Preprocess
test_person_1 = reduce_column_size(test_person_1)
convert_date_columns(test_person_1, ["birth_259D", "birthdate_87D"])
# test_person_1 = test_person_1.replace({"a55475b1": pd.NA})

# Feature selection
test_person_1 = test_person_1[person_columns]

# Feature engineering
test_person_1 = transform_location_info(test_person_1)

Static

In [ ]:
# Training
train_static = read_sets_of_dataframes(main_dir_train, ["train_static_0_0", "train_static_0_1",
                                                        "train_static_cb_0"])
train_static_a = pd.concat(train_static[:2])
train_static_b = train_static[-1]
del train_static
gc.collect()

# Test
test_static = read_sets_of_dataframes(main_dir_test, ["test_static_0_0", "test_static_0_1",
                                                      "test_static_0_2", "test_static_cb_0"])
test_static_a = pd.concat(test_static[:3])
test_static_b = test_static[-1]
del test_static
gc.collect()

Train - Static A

In [ ]:
# Preprocess
train_static_a = reduce_column_size(train_static_a)
convert_date_columns(train_static_a, ["lastapplicationdate_877D"])
# train_static_a = train_static_a.replace({"a55475b1":pd.NA})
train_static_a["lastst_736L"] = agg_categories(train_static_a, "lastst_736L")

# Feature selection
train_static_a = delete_null_columns(train_static_a, 0.4)
train_static_a = train_static_a.drop(columns=["isbidproduct_1095L", "opencred_647L"]) # only 2 categories and very imbalanced
train_static_a = train_static_a.merge(train_base[["case_id", "target"]], on=["case_id"], how="left",
                                      validate="1:1")
train_static_a = delete_constant_columns(train_static_a)
train_static_a = select_by_chi2(train_static_a)
train_static_a = train_static_a.drop(columns=["target"])

static_a_columns = train_static_a.columns

# Output
train_static_a = order_columns_alphabetically(train_static_a)
train_static_a.head()

Test - Static A

In [ ]:
test_static_a = reduce_column_size(test_static_a)
convert_date_columns(test_static_a, ["lastapplicationdate_877D"])

# Feature selection
test_static_a = test_static_a[static_a_columns]

Train - Static B

In [ ]:
# Preprocess
train_static_b = reduce_column_size(train_static_b)
convert_date_columns(train_static_b, [x for x in train_static_b.columns if "date" in x])
# train_static_b = train_static_b.replace({"a55475b1": pd.NA})

# Feature selection
train_static_b = train_static_b.drop(columns=["dateofbirth_337D"]) # already have in person
train_static_b = delete_constant_columns(train_static_b)

original_col_list = []
# we remove assignmentdate (after merge there are many nulls), birth (it's complete in person),
# pmtaverage and pmtcount (many nulls after merge), maritalst after full combination has many nulls
new_cols = ["education", "responsedate"]
for new_col in new_cols:
    original_cols = train_static_b.columns[train_static_b.columns.str.contains(new_col)].tolist()
    original_col_list.append(original_cols)
original_col_list = list(itertools.chain.from_iterable(original_col_list))
train_static_b = delete_null_columns(train_static_b, 0.4, original_col_list)

static_b_columns = train_static_b.columns

# Output
train_static_b = order_columns_alphabetically(train_static_b)
train_static_b.head()

Test - Static B

In [ ]:
# Preprocess
test_static_b = reduce_column_size(test_static_b)
convert_date_columns(test_static_b, [x for x in test_static_b.columns if "date" in x])

#
test_static_b = test_static_b[static_b_columns]

In [ ]:
# DO THIS AFTER MERGING
# from joblib import cpu_count, Parallel, delayed
# new_data = Parallel(n_jobs=cpu_count(), backend="multiprocessing")(delayed(mix_cols_parallel)(train_static_b, original_cols) for original_cols in original_col_list)
# # train_static_b_f = train_static_b[["case_id"]].copy(deep=True)
# for new_col, new_col_data in zip(new_cols, new_data):
#     train_static_b[new_col] = new_col_data

# import itertools
# original_cols = list(itertools.chain.from_iterable(original_col_list))
# train_static_b = train_static_b.drop(columns=original_cols)


# plot_null_percent(train_static_b, proportion=0.1)
# train_static_b = delete_null_columns(train_static_b, 0.1)

Debitcard - not used

Other - not used

Tax registry

In [ ]:
train_tax_registry = read_sets_of_dataframes(main_dir_train, ["train_tax_registry_a_1",
                                                              "train_tax_registry_b_1",
                                                              "train_tax_registry_c_1"])
test_tax_registry = read_sets_of_dataframes(main_dir_test, ["test_tax_registry_a_1",
                                                            "test_tax_registry_b_1",
                                                            "test_tax_registry_c_1"])
# Renaming the columns as all mean the same
tax_column_renaming = [{"amount_4527230A": "tax_deduction_amount",
                        "recorddate_4527225D": "tax_date",
                        "name_4527232M": "tax_employer_name"},
                       {"amount_4917619A": "tax_deduction_amount",
                        "deductiondate_4917603D": "tax_date",
                        "name_4917606M":"tax_employer_name"},
                       {"pmtamount_36A": "tax_deduction_amount",
                        "processingdate_168D": "tax_date",
                        "employername_160M":"tax_employer_name"}]

In [ ]:
for i in range(len(train_tax_registry)):
    train_tax_registry[i].rename(columns=tax_column_renaming[i], inplace=True)
train_tax_registry = pd.concat(train_tax_registry)

# Aggregation / feature creation
train_tax_registry = train_tax_registry.groupby(["case_id"]).agg({"tax_deduction_amount":["max","min","mean","count"]})
train_tax_registry.columns = ["_".join(x) for x in train_tax_registry.columns]
train_tax_registry = reduce_column_size(train_tax_registry)

tax_registry_columns = train_tax_registry.columns

In [ ]:
for i in range(len(test_tax_registry)):
    test_tax_registry[i].rename(columns=tax_column_renaming[i], inplace=True)
test_tax_registry = pd.concat(test_tax_registry)

test_tax_registry = test_tax_registry.groupby(["case_id"]).agg({"tax_deduction_amount":["max","min","mean","count"]})
test_tax_registry.columns = ["_".join(x) for x in test_tax_registry.columns]
test_tax_registry = reduce_column_size(test_tax_registry) 

test_tax_registry = test_tax_registry[tax_registry_columns]


Applprev

In [ ]:
train_applprev = read_sets_of_dataframes(main_dir_train, ["train_applprev_1_0",
                                                          "train_applprev_1_1",
                                                          "train_applprev_2"])
train_applprev_1 = pd.concat(train_applprev[0:2])
train_applprev_2 = train_applprev[2]
del train_applprev
gc.collect()

test_applprev = read_sets_of_dataframes(main_dir_test, ["test_applprev_1_0", "test_applprev_1_1",
                                                        "test_applprev_1_2", "test_applprev_2"])
test_applprev_1 = pd.concat(test_applprev[:3])
test_applprev_2 = test_applprev[-1]
del test_applprev
gc.collect()

Applprev 1 - Train

In [ ]:
# Preprocess
convert_date_columns(train_applprev_1, [x for x in train_applprev_1.columns if "date" in x])

# Feature selection
train_applprev_1 = delete_constant_columns(train_applprev_1)

## Deleting numerical columns with lots of nulls
numerical_data = train_applprev_1.groupby("case_id").min(numeric_only=True).reset_index()
numerical_data = delete_null_columns(numerical_data, 0.5)
applprev_1_numerical_columns = numerical_data.columns.tolist()

numerical_data = train_applprev_1[applprev_1_numerical_columns]
numerical_data = delete_constant_columns(numerical_data)
if "num_group1" in numerical_data.columns:
    numerical_data = numerical_data.drop(columns=["num_group1"])
applprev_1_numerical_columns = numerical_data.columns.tolist()

## Deleting categorical columns with lots of nulls
# categorical_data = get_dataset_by_datatype(train_applprev_1, "categorical",
#                                            extra_columns=["case_id"]).groupby(["case_id"]).count().reset_index()
# categorical_data = categorical_data.replace({0:pd.NA})
# categorical_data = delete_null_columns(categorical_data, 0.5)
# applprev_1_categorical_columns = categorical_data.columns.tolist()

# categorical_data = train_applprev_1[applprev_1_categorical_columns]
# categorical_data = categorical_data.merge(train_base[["case_id", "target"]], on=["case_id"],
#                                           how="left")
# categorical_data = select_by_chi2(categorical_data)
# categorical_data = delete_constant_columns(categorical_data)
# # filter columns with many categories
# categorical_data = categorical_data.drop(columns=["target"])
# applprev_1_categorical_columns = categorical_data.columns.tolist()



## Deleting date columns with lots of nulls
# date_data = get_dataset_by_datatype(train_applprev_1, "date",
#                                     extra_columns=["case_id"]).groupby(["case_id"]).count().reset_index()
# del date_data
# gc.collect()

del train_applprev_1
gc.collect()


# Feature engineering
numerical_data = numerical_data.groupby(["case_id"]).agg(["max", "min", "mean"])
numerical_data.columns = ["_".join(x) for x in numerical_data.columns]
numerical_data = numerical_data.reset_index()

# categorical_data = categorical_to_dummies(categorical_data)
# categorical_data = categorical_data.groupby(["case_id"]).sum()
# categorical_data = categorical_data.reset_index()


# Output
train_applprev_1 = numerical_data.copy()
del numerical_data
gc.collect()
# train_applprev_1 = train_applprev_1.merge(categorical_data, on=["case_id"], how="left")
# del categorical_data
# gc.collect()
# train_applprev_1 = train_applprev_1.merge(date_data, on=["case_id"], how="left")
# del date_data
# gc.collect()

train_applprev_1 = reduce_column_size(train_applprev_1)
train_applprev_1 = order_columns_alphabetically(train_applprev_1)

Applprev 1 - Test

In [ ]:
numerical_data = test_applprev_1[applprev_1_numerical_columns]
# categorical_data = test_applprev_1[applprev_1_categorical_columns]
# date_data = "pass"

# Feature engineering
numerical_data = numerical_data.groupby(["case_id"]).agg(["max", "min", "mean"])
numerical_data.columns = ["_".join(x) for x in numerical_data.columns]
numerical_data = numerical_data.reset_index()

# categorical_data = categorical_to_dummies(categorical_data)
# categorical_data = categorical_data.groupby(["case_id"]).sum()
# categorical_data = categorical_data.reset_index()

del test_applprev_1
gc.collect()

# Output
test_applprev_1 = numerical_data.copy()
del numerical_data
gc.collect()
# test_applprev_1 = test_applprev_1.merge(categorical_data, on=["case_id"], how="left")
# del categorical_data
# gc.collect()
# test_applprev_1 = test_applprev_1.merge(date_data, on=["case_id"], how="left")
# del date_data
# gc.collect()

test_applprev_1 = reduce_column_size(test_applprev_1)
test_applprev_1 = order_columns_alphabetically(test_applprev_1)

Applprev 2 - Train

In [ ]:
def categorize_contact(cell):
    try:
        if "MOBILE" in cell:
            return "MOBILE"
        elif "PHONE" in cell:
            return "PHONE"
        elif "EMAIL" in cell or cell in ["WHATSAPP", "SKYPE"]:
            return "EMAIL"
        else:
            return "OTHER"
    except:
        return cell

In [ ]:
# Preprocess
# train_applprev_2 = train_applprev_2.replace({"a55475b1":pd.NA})

# Feature selection
train_applprev_2 = delete_null_columns(train_applprev_2, 0.5)
train_applprev_2 = delete_constant_columns(train_applprev_2)
columns_applprev_2 = train_applprev_2.columns

# Feature engineering and aggregation
train_applprev_2 = categorical_to_dummies(train_applprev_2, dtype=64)
train_applprev_2["n_appl_2"] = train_applprev_2.groupby(["case_id"])["num_group1"].transform("count")
train_applprev_2 = train_applprev_2.groupby(["case_id"]).mean()
train_applprev_2 = train_applprev_2.drop(columns=["num_group1", "num_group2"])
train_applprev_2 = train_applprev_2.reset_index()

# Output
train_applprev_2 = reduce_column_size(train_applprev_2)
train_applprev_2 = order_columns_alphabetically(train_applprev_2)

In [ ]:
# Preprocess
# test_applprev_2 = test_applprev_2.replace({"a55475b1":pd.NA})

# Feature selection
test_applprev_2 = test_applprev_2[columns_applprev_2]

# Feature engineering
test_applprev_2 = categorical_to_dummies(test_applprev_2, dtype=64)
test_applprev_2["n_appl_2"] = test_applprev_2.groupby(["case_id"])["num_group1"].transform("count")
test_applprev_2 = test_applprev_2.groupby(["case_id"]).mean()
test_applprev_2 = test_applprev_2.reset_index()

# Output
test_applprev_2 = reduce_column_size(test_applprev_2)
test_applprev_2 = order_columns_alphabetically(test_applprev_2)

Merging everything

In [ ]:
# for dataset in [train_person_1, train_static_a, train_static_b, train_tax_registry,
#                 train_applprev_1, train_applprev_2]:
#     train_base = train_base.merge(dataset, on=["case_id"], how="left", validate="1:1")
#     del dataset
#     gc.collect()
train_base = train_base.merge(train_person_1, on=["case_id"], how="left", validate="1:1")
del train_person_1
gc.collect()
train_base = train_base.merge(train_static_a, on=["case_id"], how="left", validate="1:1")
del train_static_a
gc.collect()
train_base = train_base.merge(train_static_b, on=["case_id"], how="left", validate="1:1")
del train_static_b
gc.collect()
train_base = train_base.merge(train_tax_registry, on=["case_id"], how="left", validate="1:1")
del train_tax_registry
gc.collect()
train_base = train_base.merge(train_applprev_1, on=["case_id"], how="left", validate="1:1")
del train_applprev_1
gc.collect()
train_base = train_base.merge(train_applprev_2, on=["case_id"], how="left", validate="1:1")
del train_applprev_2
gc.collect()
train_base = train_base.merge(train_credit_bureau_a_1, on=["case_id"], how="left", validate="1:1")
del train_credit_bureau_a_1
gc.collect()
train_base = train_base.merge(train_credit_bureau_a_2, on=["case_id"], how="left", validate="1:1")
del train_credit_bureau_a_2
gc.collect()

train_base = reduce_column_size(train_base)

In [ ]:
test_base = test_base.merge(test_person_1, on=["case_id"], how="left", validate="1:1")
del test_person_1
gc.collect()
test_base = test_base.merge(test_static_a, on=["case_id"], how="left", validate="1:1")
del test_static_a
gc.collect()
test_base = test_base.merge(test_static_b, on=["case_id"], how="left", validate="1:1")
del test_static_b
gc.collect()
test_base = test_base.merge(test_tax_registry, on=["case_id"], how="left", validate="1:1")
del test_tax_registry
gc.collect()
test_base = test_base.merge(test_applprev_1, on=["case_id"], how="left", validate="1:1")
del test_applprev_1
gc.collect()
test_base = test_base.merge(test_applprev_2, on=["case_id"], how="left", validate="1:1")
del test_applprev_2
gc.collect()
test_base = test_base.merge(test_credit_bureau_a_1, on=["case_id"], how="left", validate="1:1")
del test_credit_bureau_a_1
gc.collect()
test_base = test_base.merge(test_credit_bureau_a_2, on=["case_id"], how="left", validate="1:1")
del test_credit_bureau_a_2
gc.collect()

test_base = reduce_column_size(test_base)

In [ ]:
# original_col_list = []
# new_cols = ["education", "responsedate"]

# for new_col in new_cols:
#     original_cols = train_base.columns[train_base.columns.str.contains(new_col)].tolist()
#     original_col_list.append(original_cols)


# DO THIS AFTER MERGING
# from joblib import cpu_count, Parallel, delayed
# new_data = Parallel(n_jobs=cpu_count(), backend="multiprocessing")(delayed(mix_cols_parallel)(train_base, original_cols) for original_cols in original_col_list)
# for new_col, new_col_data in zip(new_cols, new_data):
#     train_base[new_col] = new_col_data

# import itertools
# original_cols = list(itertools.chain.from_iterable(original_col_list))
# train_base = train_base.drop(columns=original_cols)

In [ ]:
def add_age_features(data: pd.DataFrame, birthdate_variable: str = "birth_259D", drop_original_cols=False):
    date_variables = data.columns[data.dtypes == "datetime64[ns]"]
    date_variables = date_variables.difference([birthdate_variable])
    for variable in date_variables:
        age = (data[variable] - data[birthdate_variable]).dt.days
        age //= 365 # convert to full years
        data[f"age_at_{variable}"] = age
    if drop_original_cols:
        data.drop(columns=date_variables.tolist()+[birthdate_variable],
                  inplace=True)

add_age_features(train_base, drop_original_cols=True)
add_age_features(test_base, drop_original_cols=True)

In [ ]:
# Need to aggregate similar columns and also categories
cut_category_cols = select_categorical_with_many_categories(train_base, 20)

train_base = train_base.drop(columns=cut_category_cols)
test_base = test_base.drop(columns=cut_category_cols)

In [ ]:
# Treating the rest of categorical features
train_base = categorical_to_dummies(train_base)
test_base = categorical_to_dummies(test_base)
test_base = test_base.reindex(columns=train_base.columns)
test_base = test_base[train_base.columns]

In [ ]:
# Removing higly correlated features
print(train_base.shape)
# train_base = remove_highly_correlated_features(train_base)
print(train_base.shape)
test_base = test_base[train_base.columns]

In [ ]:
train_base.shape

In [ ]:
# Deleting duplicate columns
# train_base.T.drop_duplicates().T

In [ ]:
# Deleting very correlated columns
# numerical_cols = [x for x in train_base.columns if train_base[x].dtype in ["float64", "int64"]]
# numerical_cols = [x for x in numerical_cols if x not in ['case_id', 'WEEK_NUM', 'target', 'num_group1', 'num_group2']]
# numerical_base = train_base[numerical_cols]#.fillna(0)
# linear_corrs = numerical_base.corr()

# linear_corrs = linear_corrs.unstack().reset_index()
# linear_corrs.columns = ["v1", "v2", "corr"]
# def reorder_corr_values(row: pd.DataFrame):
#     if row["v1"] > row["v2"]:
#         row["v1"], row["v2"] = row["v2"], row["v1"]
#     return row
# linear_corrs = linear_corrs.apply(reorder_corr_values, axis=1)
# linear_corrs = linear_corrs.drop_duplicates()
# linear_corrs = linear_corrs[linear_corrs["v1"]!=linear_corrs["v2"]]
# linear_corrs = linear_corrs[linear_corrs["corr"]>0.8]


# train_base = train_base.drop(columns=linear_corrs["v2"].unique())

tasks:
- check for columns in the original datasets that vary with target = 1 (we exclude them as a result of deleting null values)
- correlation stuff

In [ ]:
# i dont have enough memory to execute this (not even 1 column)
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# numerical_cols = [x for x in train_base.columns if train_base[x].dtype in ["float64", "int64"]]
# numerical_base = train_base[numerical_cols].fillna(0)
# vif_scores = [variance_inflation_factor(numerical_base.values, feature) for feature in range(len(numerical_base.columns))]

Model

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score 
def gini_stability(base):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", "score"]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", "score"]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x["score"])-1).tolist()
    
    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + 88.0 * min(0, a) - 0.5 * res_std

In [ ]:
import lightgbm
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 3,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "n_estimators": 2000,
    # "n_estimators": 5000,
    "verbose": -1,
    "deterministic": True,
}
model = lightgbm.LGBMClassifier(**params)

train_base_cols = ["case_id", "num_group1", "num_group2", "date_decision",
                   "MONTH", "WEEK_NUM", "target", "score"]
train_base_cols = [x for x in train_base_cols if x in train_base.columns]

# categorical_cols = []
# for col in train_base.columns:
#     if train_base[col].dtype == "object":
#         # train_base = pd.concat([train_base, pd.get_dummies(train_base[col], dtype="int64", drop_first=False)],
#         #                            axis=1)
#         categorical_cols.append(col)
# train_base = train_base.drop(columns=categorical_cols)

In [ ]:
model.fit(train_base.drop(columns=train_base_cols),
          train_base["target"])
# 22 segundos en hacer el fit de 74 variables

In [ ]:
preds = model.predict_proba(train_base.drop(columns=train_base_cols))
train_base["score"] = preds[:,1] # probability of defaulting
print(gini_stability(train_base))
train_base.drop(columns=["score"], inplace=True)
# 0.5761124458701475 without fillna

In [ ]:
# from sklearn.feature_selection import mutual_info_classif
# mi = mutual_info_classif(train_base.drop(columns=train_base_cols).fillna(0), train_base["target"])
# 10 mins -- 74 features

In [ ]:
features, importances = train_base.drop(columns=train_base_cols).columns.tolist(), model.feature_importances_.tolist()
feature_importances = sorted([(y,x) for x,y in zip(features, importances)], reverse=True)
feature_importances

In [ ]:
test_preds = model.predict_proba(test_base.drop(columns=train_base_cols))
#test_preds = model.predict_proba(test_base[train_base.drop(columns=train_base_cols).columns])

submission = test_base[["case_id"]]
submission["score"] = test_preds[:,1]
submission.set_index("case_id")
submission.to_csv("./submission.csv")